# SAADE v5 · 03 — Аудіо, усі сценарії та результати моделей

Перегляд готових результатів **без повторного навчання**. Якщо моделей ще не навчали, можна прослухати й порівняти акустичні сценарії; таблиці E1/E2/E3 з’являться після основного експерименту.

## 0. Підключення (типово PILOT; для matched змініть OUT на SAADE_AUA_2026_v5_FINAL)

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys, json, shutil, importlib.util
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path('/home/jovyan/work/audio_real_fake')
AUA = PROJECT / 'AUA'
RESULTS = PROJECT / 'results'
OUT = RESULTS / 'SAADE_AUA_2026_v5_PILOT'
assert AUA.is_dir(), f'Не знайдено AUA: {AUA}'
assert shutil.which('ffmpeg'), 'Потрібно встановити ffmpeg, щоб відтворити MP3-сценарії'
sys.path.insert(0,str(Path.cwd().resolve()))
import saade_final_core as S
import saade_final_data as D
import saade_final_methods as M
import saade_final_visuals as V
import saade_final_adaptive as A
for mod in (S,D,M,V,A):
    assert Path(mod.__file__).resolve().parent == Path.cwd().resolve(), f'Чужа версія модуля: {mod.__file__}'
cfg=S.Config(root=str(AUA),out=str(OUT),scenario_profile='v2_5level',
             seed=42,seconds=4.,min_seconds=2.,augmentation_probability=.50,
             batch_size=12,lr=.0005,epochs=15,use_cache=True)
print('AUA:',AUA,'\nRESULTS:',OUT)
print('Дані лише для читання. Melnyk_dus не використовується.')

## 1. Склад даних, незалежність і класи

In [ ]:
SPLITS=OUT/'V5_02_LOCKED_EXPERIMENT_SEGMENTS.csv'
assert SPLITS.exists(),'Спочатку виконайте 02_Full_Dissertation_Experiment.ipynb хоча б до етапу 4'
splits=pd.read_csv(SPLITS,low_memory=False)
display(splits.groupby(['split','source','label']).agg(
     fragments=('path','size'),parents=('parent_file_id','nunique'),groups=('group_id','nunique')).reset_index())
print('Сценарії:');display(V.all_scenario_metadata(cfg))

## 2. Прослуховування 21 сценарію для REAL і FAKE

In [ ]:
examples=V.pick_examples(splits,'validation')
for item in examples:
    V.show_audio_gallery(item,cfg,tag='v5_viewer',show_players=True,show_figures=True,max_players=None)
V.make_interactive_player(splits,cfg,'validation')

## 3. Прогнози та ASRI за всіма акустичними сценаріями

In [ ]:
path=OUT/'V5_04_VALIDATION_PREDICTIONS_ALL.csv'
if not path.exists():
    print('Немає прогнозів: навчання не виконано. Увімкніть RUN_TRAINING=True в основному ноутбуці.')
else:
    pred=pd.read_csv(path)
    display(pred.groupby(['seed','model','regime','category','level']).size().rename('n').reset_index().head(35))
    for _seed,_pred in pred.groupby('seed'):
        V.report_scenario_tests(_pred,cfg,tag=f'v5_viewer_seed{_seed}',show=True)
    for file in ['V5_05_ASRI_BY_SEED.csv','V5_05_PAIRED_SEED_DELTAS.csv',
                 'V5_05_BOOTSTRAP_STRATIFIED_VOICES.csv','V5_05_DELTA_SUMMARY.csv']:
        f=OUT/file
        if f.exists():
            print(file);display(pd.read_csv(f))

## 4. Журнал адаптації

In [ ]:
hist=list((OUT/'models').glob('*_adaptive_seed*_policy_history.csv')) if (OUT/'models').exists() else []
print('Файлів із динамічними вагами:',len(hist))
for p in hist[:3]:
    print(p.name)
    d=pd.read_csv(p)
    display(d.head(10))